# Chapter 3: Introduction to Reinforcement Learning — Teaching Agents to Decide

> **Book:** Agentic AI Engineering  
> **Part:** 2 — RL Methods  
> **System:** Guardian Angel System (GAS) — Multi-agent diabetic patient supervision

In [ ]:
# Install dependencies
# pip install gymnasium numpy matplotlib

In [ ]:
import os
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"
print(f"TEST_MODE: {TEST_MODE}")

---
## 🎯 1. Chapter Goal

By the end of this notebook you will:
- Understand the **MDP framework**: states, actions, rewards, policies
- Implement **Q-learning** from scratch
- Train an agent on a **GridWorld** environment
- Build a **GAS-inspired glucose management** RL environment
- Visualize learned Q-values and learning curves

---
## 📖 2. Theory Recap

### 2.1 Markov Decision Process (MDP)

An MDP is defined by the tuple **(S, A, R, P, γ)**:
- **S**: State space (e.g., glucose level)
- **A**: Action space (e.g., insulin dose)
- **R**: Reward function (e.g., +1 for time-in-range)
- **P**: Transition probabilities P(s'|s,a)
- **γ**: Discount factor (0 < γ < 1)

### 2.2 Bellman Equation

The optimal Q-function satisfies:
$$Q^*(s,a) = R(s,a) + \gamma \max_{a'} Q^*(s', a')$$

### 2.3 Q-Learning Update

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]$$

Where α is the learning rate and the term in brackets is the **TD error**.

### 2.4 Exploration vs Exploitation

**ε-greedy policy:**
- With probability ε: take random action (explore)
- With probability 1-ε: take best known action (exploit)
- Decay ε over time: start exploring, gradually exploit

### 2.5 Key RL Concepts for GAS

| RL Concept | GAS Meaning |
|-----------|-------------|
| State | Patient glucose, IOB, meal context, time of day |
| Action | Insulin dose (0, 0.5, 1, 2, 4 units) |
| Reward | +1 if glucose in range (70-180), -10 if hypoglycemia |
| Episode | One day of patient monitoring |
| Policy | Advisor Agent's recommendation strategy |

---
## 🔨 3. Build It

### 3.1 GridWorld Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import defaultdict

class GridWorld:
    """
    Simple 5x5 GridWorld environment.
    
    Grid layout:
    S . . . .
    . X . X .
    . . . . .
    . X . X .
    . . . . G
    
    S = Start, G = Goal, X = Wall, . = Empty
    """
    
    def __init__(self, size=5):
        self.size = size
        self.start = (0, 0)
        self.goal = (size-1, size-1)
        self.walls = {(1,1), (1,3), (3,1), (3,3)}
        self.actions = [0, 1, 2, 3]  # up, down, left, right
        self.action_names = ['↑', '↓', '←', '→']
        self.action_deltas = [(-1,0), (1,0), (0,-1), (0,1)]
        self.reset()
    
    def reset(self):
        self.state = self.start
        return self.state
    
    def step(self, action):
        dr, dc = self.action_deltas[action]
        new_r = self.state[0] + dr
        new_c = self.state[1] + dc
        
        # Check bounds and walls
        if (0 <= new_r < self.size and 0 <= new_c < self.size 
                and (new_r, new_c) not in self.walls):
            self.state = (new_r, new_c)
        
        done = self.state == self.goal
        reward = 10.0 if done else -0.1  # small penalty per step
        return self.state, reward, done
    
    def render(self, q_table=None):
        """Render the grid with optional Q-value arrows."""
        grid = np.zeros((self.size, self.size))
        for wall in self.walls:
            grid[wall] = -1
        grid[self.goal] = 2
        grid[self.state] = 1
        return grid


# Test the environment
env = GridWorld()
state = env.reset()
print(f"Initial state: {state}")
state, reward, done = env.step(1)  # move down
print(f"After moving down: state={state}, reward={reward}, done={done}")
print(f"Action space: {env.action_names}")
print(f"Grid size: {env.size}x{env.size}")
print(f"Walls at: {env.walls}")

In [ ]:
class QLearningAgent:
    """
    Q-Learning agent with ε-greedy exploration.
    """
    
    def __init__(self, n_states_r, n_states_c, n_actions, 
                 alpha=0.1, gamma=0.99, epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01):
        self.n_actions = n_actions
        self.alpha = alpha      # learning rate
        self.gamma = gamma      # discount factor
        self.epsilon = epsilon  # exploration rate
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        
        # Q-table: state (r,c) → action values
        self.q_table = defaultdict(lambda: np.zeros(n_actions))
    
    def select_action(self, state):
        """ε-greedy action selection."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)  # explore
        return np.argmax(self.q_table[state])  # exploit
    
    def update(self, state, action, reward, next_state, done):
        """Q-learning update rule."""
        current_q = self.q_table[state][action]
        
        if done:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.q_table[next_state])
        
        # TD update
        td_error = target - current_q
        self.q_table[state][action] += self.alpha * td_error
        return abs(td_error)
    
    def decay_epsilon(self):
        """Decay exploration rate."""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


def train_q_learning(env, agent, n_episodes=500, max_steps=100):
    """Train Q-learning agent and return metrics."""
    episode_rewards = []
    episode_lengths = []
    td_errors = []
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        steps = 0
        episode_td_errors = []
        
        for step in range(max_steps):
            action = agent.select_action(state)
            next_state, reward, done = env.step(action)
            
            td_error = agent.update(state, action, reward, next_state, done)
            episode_td_errors.append(td_error)
            
            state = next_state
            total_reward += reward
            steps += 1
            
            if done:
                break
        
        agent.decay_epsilon()
        episode_rewards.append(total_reward)
        episode_lengths.append(steps)
        td_errors.append(np.mean(episode_td_errors))
    
    return episode_rewards, episode_lengths, td_errors


# Train!
np.random.seed(42)
env = GridWorld()
agent = QLearningAgent(env.size, env.size, len(env.actions))

n_episodes = 300 if not TEST_MODE else 50
print(f"Training Q-learning agent for {n_episodes} episodes...")
rewards, lengths, td_errors = train_q_learning(env, agent, n_episodes=n_episodes)

print(f"\nTraining complete!")
print(f"Final epsilon: {agent.epsilon:.3f}")
print(f"Last 50 episodes avg reward: {np.mean(rewards[-50:]):.2f}")
print(f"Last 50 episodes avg length: {np.mean(lengths[-50:]):.1f} steps")

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

window = 20

# Episode rewards
ax1 = axes[0]
ax1.plot(rewards, alpha=0.3, color='blue', label='Raw')
smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
ax1.plot(range(window-1, len(rewards)), smoothed, color='blue', linewidth=2, label=f'{window}-ep avg')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.set_title('Learning Curve: Episode Reward', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Episode lengths
ax2 = axes[1]
ax2.plot(lengths, alpha=0.3, color='green', label='Raw')
smoothed_len = np.convolve(lengths, np.ones(window)/window, mode='valid')
ax2.plot(range(window-1, len(lengths)), smoothed_len, color='green', linewidth=2, label=f'{window}-ep avg')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Steps to Goal')
ax2.set_title('Learning Curve: Episode Length', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Q-value heatmap for best action
ax3 = axes[2]
q_grid = np.zeros((env.size, env.size))
best_action_grid = np.zeros((env.size, env.size), dtype=int)

for r in range(env.size):
    for c in range(env.size):
        state = (r, c)
        if state in env.walls:
            q_grid[r, c] = -5
        else:
            q_vals = agent.q_table[state]
            q_grid[r, c] = np.max(q_vals)
            best_action_grid[r, c] = np.argmax(q_vals)

im = ax3.imshow(q_grid, cmap='RdYlGn', aspect='auto')
plt.colorbar(im, ax=ax3, label='Max Q-value')

# Add arrows for best actions
arrow_map = {0: '↑', 1: '↓', 2: '←', 3: '→'}
for r in range(env.size):
    for c in range(env.size):
        if (r, c) not in env.walls:
            arrow = arrow_map[best_action_grid[r, c]]
            ax3.text(c, r, arrow, ha='center', va='center', fontsize=14, fontweight='bold')

# Mark start and goal
ax3.text(0, 0, 'S', ha='center', va='center', fontsize=12, color='blue', fontweight='bold')
ax3.text(env.size-1, env.size-1, 'G', ha='center', va='center', fontsize=12, color='red', fontweight='bold')
ax3.set_title('Learned Q-values & Policy\n(arrows = best action)', fontweight='bold')
ax3.set_xticks(range(env.size))
ax3.set_yticks(range(env.size))

plt.tight_layout()
plt.savefig('q_learning_results.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 GAS GridWorld — Glucose Management RL Environment

In [ ]:
class GlucoseManagementEnv:
    """
    Simplified glucose management environment for RL.
    
    State: glucose level (discretized into bins)
    Action: insulin dose (0, 0.5, 1, 2, 4 units)
    Reward: +1 if in range (70-180), -10 if hypoglycemia (<54), -1 if hyperglycemia (>250)
    
    Glucose dynamics (simplified):
    - Natural rise: +5-15 mg/dL per step (meal absorption)
    - Insulin effect: -20 mg/dL per unit (delayed)
    - Random noise: ±10 mg/dL
    """
    
    # Glucose bins: <54, 54-70, 70-100, 100-140, 140-180, 180-250, >250
    GLUCOSE_BINS = [0, 54, 70, 100, 140, 180, 250, 400]
    BIN_NAMES = ['<54\n(Severe Hypo)', '54-70\n(Hypo)', '70-100\n(Low Normal)', 
                 '100-140\n(Normal)', '140-180\n(High Normal)', '180-250\n(Hyper)', '>250\n(Severe Hyper)']
    
    ACTIONS = [0, 0.5, 1.0, 2.0, 4.0]  # insulin units
    ACTION_NAMES = ['0U', '0.5U', '1U', '2U', '4U']
    
    def __init__(self):
        self.n_states = len(self.GLUCOSE_BINS) - 1  # 7 bins
        self.n_actions = len(self.ACTIONS)
        self.reset()
    
    def _glucose_to_state(self, glucose):
        """Convert continuous glucose to discrete state."""
        for i in range(len(self.GLUCOSE_BINS) - 1):
            if glucose < self.GLUCOSE_BINS[i+1]:
                return i
        return len(self.GLUCOSE_BINS) - 2
    
    def reset(self):
        # Start in normal range
        self.glucose = np.random.uniform(100, 160)
        self.step_count = 0
        self.iob = 0.0  # insulin on board
        return self._glucose_to_state(self.glucose)
    
    def step(self, action_idx):
        insulin_dose = self.ACTIONS[action_idx]
        self.step_count += 1
        
        # Glucose dynamics
        meal_rise = np.random.uniform(5, 20)  # natural rise
        insulin_effect = (insulin_dose + self.iob * 0.3) * 20  # insulin lowers glucose
        noise = np.random.normal(0, 8)
        
        self.glucose = self.glucose + meal_rise - insulin_effect + noise
        self.glucose = np.clip(self.glucose, 30, 400)
        
        # Update IOB (insulin on board decays)
        self.iob = max(0, self.iob * 0.7 + insulin_dose * 0.3)
        
        # Reward function
        if self.glucose < 54:
            reward = -10.0  # severe hypoglycemia — dangerous!
        elif self.glucose < 70:
            reward = -3.0   # mild hypoglycemia
        elif self.glucose <= 180:
            reward = +1.0   # in target range
        elif self.glucose <= 250:
            reward = -0.5   # mild hyperglycemia
        else:
            reward = -2.0   # severe hyperglycemia
        
        done = self.step_count >= 288  # 24 hours (5-min intervals)
        state = self._glucose_to_state(self.glucose)
        
        return state, reward, done, {'glucose': self.glucose}


# Train Q-learning on glucose environment
np.random.seed(42)
glucose_env = GlucoseManagementEnv()
glucose_agent = QLearningAgent(
    glucose_env.n_states, 1, glucose_env.n_actions,
    alpha=0.1, gamma=0.95, epsilon=1.0, epsilon_decay=0.998
)

n_episodes = 500 if not TEST_MODE else 50
print(f"Training glucose management agent for {n_episodes} episodes...")

glucose_rewards = []
time_in_range = []

for episode in range(n_episodes):
    state = glucose_env.reset()
    total_reward = 0
    in_range_steps = 0
    total_steps = 0
    
    done = False
    while not done:
        action = glucose_agent.select_action((state, 0))  # use (state, 0) as key
        next_state, reward, done, info = glucose_env.step(action)
        glucose_agent.update((state, 0), action, reward, (next_state, 0), done)
        
        if 70 <= info['glucose'] <= 180:
            in_range_steps += 1
        total_steps += 1
        total_reward += reward
        state = next_state
    
    glucose_agent.decay_epsilon()
    glucose_rewards.append(total_reward)
    time_in_range.append(in_range_steps / total_steps * 100)

print(f"\nFinal 50-episode avg Time-in-Range: {np.mean(time_in_range[-50:]):.1f}%")
print(f"(Clinical target: >70% TIR)")

In [ ]:
# Visualize glucose RL results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

window = 30

# Time in range over training
ax1 = axes[0]
ax1.plot(time_in_range, alpha=0.3, color='green')
smoothed_tir = np.convolve(time_in_range, np.ones(window)/window, mode='valid')
ax1.plot(range(window-1, len(time_in_range)), smoothed_tir, color='green', linewidth=2)
ax1.axhline(y=70, color='red', linestyle='--', label='Clinical target (70%)')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Time in Range (%)')
ax1.set_title('Glucose Management Learning\nTime-in-Range (70-180 mg/dL)', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 100)

# Q-values heatmap: state × action
ax2 = axes[1]
q_matrix = np.zeros((glucose_env.n_states, glucose_env.n_actions))
for s in range(glucose_env.n_states):
    q_matrix[s] = glucose_agent.q_table.get((s, 0), np.zeros(glucose_env.n_actions))

im = ax2.imshow(q_matrix, cmap='RdYlGn', aspect='auto')
plt.colorbar(im, ax=ax2, label='Q-value')
ax2.set_xticks(range(glucose_env.n_actions))
ax2.set_xticklabels(glucose_env.ACTION_NAMES)
ax2.set_yticks(range(glucose_env.n_states))
ax2.set_yticklabels(glucose_env.BIN_NAMES, fontsize=7)
ax2.set_xlabel('Insulin Action')
ax2.set_ylabel('Glucose State')
ax2.set_title('Learned Q-values\n(State × Action)', fontweight='bold')

# Simulate one episode with trained policy
ax3 = axes[2]
state = glucose_env.reset()
glucose_history = [glucose_env.glucose]
action_history = []

# Use greedy policy (epsilon=0)
old_epsilon = glucose_agent.epsilon
glucose_agent.epsilon = 0

done = False
step = 0
while not done and step < 100:  # show first 100 steps
    action = glucose_agent.select_action((state, 0))
    next_state, reward, done, info = glucose_env.step(action)
    glucose_history.append(info['glucose'])
    action_history.append(glucose_env.ACTIONS[action])
    state = next_state
    step += 1

glucose_agent.epsilon = old_epsilon

time_steps = range(len(glucose_history))
ax3.plot(time_steps, glucose_history, 'b-', linewidth=2, label='Glucose')
ax3.axhspan(70, 180, alpha=0.2, color='green', label='Target range')
ax3.axhline(y=70, color='orange', linestyle='--', alpha=0.7)
ax3.axhline(y=180, color='orange', linestyle='--', alpha=0.7)
ax3.set_xlabel('Time Step (5-min intervals)')
ax3.set_ylabel('Glucose (mg/dL)')
ax3.set_title('Trained Agent: Glucose Trajectory\n(Greedy Policy)', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('glucose_rl_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSimulated episode TIR: {sum(70 <= g <= 180 for g in glucose_history)/len(glucose_history)*100:.1f}%")

---
## 🏥 4. GAS Connection

The GAS Advisor Agent's recommendation policy is essentially a learned mapping from patient state to action. RL gives us the framework to think about and train this policy.

**MDP formulation for GAS:**

```
State Space:
  s = (glucose, trend, iob, meal_context, time_of_day, hba1c)
  
Action Space:
  a = (insulin_dose, carb_recommendation, alert_level)
  
Reward Function:
  r = +1.0  if 70 ≤ glucose ≤ 180  (time in range)
  r = -10.0 if glucose < 54         (severe hypoglycemia — DANGEROUS)
  r = -3.0  if 54 ≤ glucose < 70    (mild hypoglycemia)
  r = -0.5  if 180 < glucose ≤ 250  (mild hyperglycemia)
  r = -2.0  if glucose > 250         (severe hyperglycemia)
  
Episode:
  One day of patient monitoring (288 steps × 5 minutes)
  
Policy:
  π(a|s) = Advisor Agent's LLM (fine-tuned with RL)
```

**Key insight:** The asymmetric reward (hypoglycemia = -10 vs hyperglycemia = -2) reflects the medical reality that low blood sugar is immediately dangerous, while high blood sugar causes long-term damage. This shapes the agent to be conservative with insulin dosing.

---
## ✅ 5. Checkpoint

### What We Built

| Component | Description |
|-----------|-------------|
| `GridWorld` | 5×5 grid navigation environment |
| `QLearningAgent` | Q-learning with ε-greedy exploration |
| `GlucoseManagementEnv` | GAS-inspired glucose RL environment |
| Learning curves | Episode reward and TIR over training |
| Q-value heatmap | Visualized learned policy |

### Key Takeaways

1. **Q-learning converges** to the optimal policy in tabular settings
2. **Asymmetric rewards** encode medical priorities (hypoglycemia >> hyperglycemia)
3. **Time-in-Range (TIR)** is the key metric for glucose management
4. **ε-greedy exploration** is essential — pure exploitation gets stuck in local optima

### What Comes Next

**Chapter 4: RL for Language Models** — We'll apply RL concepts to LLMs, starting with Supervised Fine-Tuning (SFT) as the foundation before adding RL-based alignment.

```
Ch3 ✅ (RL Basics) → Ch4 (SFT) → Ch5 (PPO) → Ch6 (DPO) → Ch7 (GRPO)
```